<a href="https://colab.research.google.com/github/luciacardozo472/TRABAJOP_IA/blob/main/multiagentes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Instalacion de dependencias

In [ ]:
!pip install -q transformers sentence-transformers scikit-learn pandas numpy torch

Dataset CSV - Empleados (sin normalizar)

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 500

departamentos = ['IT', 'Ventas', 'RRHH', 'Gerencia', 'Marketing', 'Finanzas']
satisfacciones = ['alta', 'media', 'baja']

edades = np.random.randint(22, 60, n)
experiencias = np.random.randint(0, 35, n)
departamento = np.random.choice(departamentos, n)

# Salario correlacionado con experiencia y departamento
salario_base = {'IT': 55000, 'Ventas': 38000, 'RRHH': 42000,
                'Gerencia': 90000, 'Marketing': 45000, 'Finanzas': 60000}
salarios = np.array([salario_base[d] + experiencias[i] * 1500 + np.random.randint(-5000, 5000)
                     for i, d in enumerate(departamento)])

# Satisfaccion correlacionada con salario
satisfaccion = []
for s in salarios:
    if s > 75000:
        satisfaccion.append(np.random.choice(['alta', 'media'], p=[0.75, 0.25]))
    elif s > 50000:
        satisfaccion.append(np.random.choice(['alta', 'media', 'baja'], p=[0.4, 0.4, 0.2]))
    else:
        satisfaccion.append(np.random.choice(['media', 'baja'], p=[0.35, 0.65]))

df_raw = pd.DataFrame({
    'edad': edades,
    'salario': salarios,
    'departamento': departamento,
    'experiencia': experiencias,
    'satisfaccion': satisfaccion
})

# Introducir valores nulos (~5%)
for col in ['salario', 'experiencia', 'satisfaccion']:
    idx = np.random.choice(df_raw.index, size=int(n * 0.05), replace=False)
    df_raw.loc[idx, col] = np.nan

print(f'Dataset generado: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas')
print(f'Valores nulos: {df_raw.isnull().sum().sum()}')
print(f'Distribucion satisfaccion:')
print(df_raw['satisfaccion'].value_counts())
df_raw.head(10)

Dataset generado: 500 filas x 5 columnas
Valores nulos: 75
Distribucion satisfaccion:
satisfaccion
alta     263
media    153
baja      59
Name: count, dtype: int64


,edad,salario,departamento,experiencia,satisfaccion
0,50,57488.0,Finanzas,1.0,alta
1,36,43116.0,Ventas,4.0,baja
2,29,81446.0,Ventas,28.0,media
3,42,81179.0,IT,18.0,alta
4,40,52182.0,Marketing,7.0,alta
5,44,42626.0,RRHH,0.0,media
6,32,78924.0,Marketing,21.0,alta
7,32,61933.0,RRHH,16.0,alta
8,45,NaN,Gerencia,6.0,alta
9,57,77980.0,Marketing,24.0,media


---
## AGENTE 1 - Normalizador

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

class AgenteNormalizador:
    def __init__(self):
        self.scaler = StandardScaler()
        self.label_encoders = {}
        self.log = []

    def limpiar(self, df):
        nulos = df.isnull().sum().sum()
        df = df.drop_duplicates()
        self.log.append(f'[Limpieza] Nulos encontrados: {nulos}')
        return df

    def imputar(self, df):
        for col in df.columns:
            if df[col].isnull().any():
                if df[col].dtype in ['float64', 'int64']:
                    val = df[col].median()
                    df[col] = df[col].fillna(val)
                    self.log.append(f'[Imputacion] {col} -> mediana ({val:.1f})')
                else:
                    val = df[col].mode()[0]
                    df[col] = df[col].fillna(val)
                    self.log.append(f'[Imputacion] {col} -> moda ({val})')
        return df

    def codificar(self, df, cols):
        for col in cols:
            le = LabelEncoder()
            df[col + '_enc'] = le.fit_transform(df[col].astype(str))
            self.label_encoders[col] = le
            self.log.append(f'[Codificacion] {col} -> clases: {list(le.classes_)}')
        return df

    def escalar(self, df, cols):
        df[cols] = self.scaler.fit_transform(df[cols])
        self.log.append(f'[Escalado] StandardScaler en: {cols}')
        return df

    def procesar(self, df):
        print('=' * 55)
        print('AGENTE 1 - Normalizador')
        print('=' * 55)
        df = self.limpiar(df.copy())
        df = self.imputar(df)
        df = self.codificar(df, ['departamento', 'satisfaccion'])
        df = self.escalar(df, ['edad', 'salario', 'experiencia'])
        for e in self.log:
            print(' ', e)
        print(f'\nDataset limpio: {df.shape}')
        return df

agente1 = AgenteNormalizador()
df_limpio = agente1.procesar(df_raw)
df_limpio[['edad', 'salario', 'experiencia', 'departamento_enc', 'satisfaccion_enc']].head()

AGENTE 1 - Normalizador
  [Limpieza] Nulos encontrados: 75
  [Imputacion] salario -> mediana (78791.0)
  [Imputacion] experiencia -> mediana (18.0)
  [Imputacion] satisfaccion -> moda (alta)
  [Codificacion] departamento -> clases: ['Finanzas', 'Gerencia', 'IT', 'Marketing', 'RRHH', 'Ventas']
  [Codificacion] satisfaccion -> clases: [np.str_('alta'), np.str_('baja'), np.str_('media')]
  [Escalado] StandardScaler en: ['edad', 'salario', 'experiencia']

Dataset limpio: (500, 7)


,edad,salario,experiencia,departamento_enc,satisfaccion_enc
0,0.786010,-0.964922,-1.612906,0,0
1,-0.482040,-1.619880,-1.314441,5,1
2,-1.116065,0.126886,1.073281,5,2
3,0.061410,0.114719,0.078397,2,0
4,-0.119740,-1.206726,-1.015976,3,0


---
## AGENTE 2 - Entrenador

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

class AgenteEntrenador:
    def __init__(self, modelo_emb='all-MiniLM-L6-v2'):
        print('=' * 55)
        print('AGENTE 2 - Entrenador')
        print('=' * 55)
        print(f'  Cargando embedder: {modelo_emb}')
        self.embedder = SentenceTransformer(modelo_emb)
        self.mejor_modelo = None
        self.mejor_score = 0
        self.metricas = {}

    def generar_embeddings(self, df_orig):
        print('\n  [Embeddings] Generando representaciones semanticas...')
        textos = df_orig.apply(
            lambda r: f"empleado de {r['departamento']}, satisfaccion {r['satisfaccion']}, experiencia aproximada",
            axis=1
        ).tolist()
        emb = self.embedder.encode(textos, show_progress_bar=True)
        print(f'  [Embeddings] Shape: {emb.shape}')
        return emb

    def construir_features(self, df, emb):
        cols = ['edad', 'salario', 'experiencia', 'departamento_enc']
        X = np.hstack([df[cols].values, emb])
        print(f'  [Features] {len(cols)} numericas + {emb.shape[1]} embedding = {X.shape[1]} total')
        return X

    def entrenar(self, X, y):
        print('\n  [Entrenamiento] Validacion cruzada cv=5:')
        candidatos = {
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
            'Logistic Regression': LogisticRegression(max_iter=500, random_state=42)
        }
        for nombre, modelo in candidatos.items():
            scores = cross_val_score(modelo, X, y, cv=5, scoring='accuracy')
            m, s = scores.mean(), scores.std()
            self.metricas[nombre] = {'accuracy_media': round(m, 4), 'std': round(s, 4)}
            print(f'    {nombre}: {m:.4f} +/- {s:.4f}')
            if m > self.mejor_score:
                self.mejor_score = m
                self.mejor_nombre = nombre
                self.mejor_modelo = modelo
        self.mejor_modelo.fit(X, y)
        print(f'\n  Mejor modelo: {self.mejor_nombre} ({self.mejor_score:.4f})')

    def procesar(self, df, df_orig):
        emb = self.generar_embeddings(df_orig)
        X = self.construir_features(df, emb)
        y = df['satisfaccion_enc'].values
        self.entrenar(X, y)
        return {
            'modelo': self.mejor_modelo,
            'nombre_modelo': self.mejor_nombre,
            'metricas': self.metricas,
            'mejor_accuracy': self.mejor_score,
            'X': X, 'y': y
        }

agente2 = AgenteEntrenador()
resultado = agente2.procesar(df_limpio, df_raw)

AGENTE 2 - Entrenador
  Cargando embedder: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


  [Embeddings] Generando representaciones semanticas...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

  [Embeddings] Shape: (500, 384)
  [Features] 4 numericas + 384 embedding = 388 total

  [Entrenamiento] Validacion cruzada cv=5:
    Random Forest: 1.0000 +/- 0.0000
    Gradient Boosting: 1.0000 +/- 0.0000
    Logistic Regression: 0.9900 +/- 0.0089

  Mejor modelo: Random Forest (1.0000)



AGENTE 3 - Comunicador

In [ ]:
from transformers import pipeline

class AgenteComunicador:
    def __init__(self):
        print('=' * 55)
        print('AGENTE 3 - Comunicador')
        print('=' * 55)
        print('  Cargando modelo de generacion de texto...')
        self.generador = pipeline(
            'text-generation',
            model='sshleifer/tiny-gpt2',
            max_new_tokens=80
        )

    def generar_reporte(self, resultado, df_original):
        mejor = resultado['nombre_modelo']
        acc = resultado['mejor_accuracy']
        metricas = resultado['metricas']

        ctx = (f'ML report: dataset with {df_original.shape[0]} employees. '
               f'Best model: {mejor}, accuracy: {acc:.4f}. Summary:')
        print('  [Generando reporte...]')
        salida = self.generador(ctx, do_sample=False)[0]['generated_text']

        dist = df_original['satisfaccion'].value_counts().to_dict()

        print('\n' + '=' * 55)
        print('REPORTE FINAL - Agente 3 Comunicador')
        print('=' * 55)
        print(f'Dataset analizado: {df_original.shape[0]} registros, {df_original.shape[1]} variables.')
        print(f'Valores faltantes tratados: {df_original.isnull().sum().sum()} celdas.')
        print(f'Distribucion: alta={dist.get("alta",0)}, media={dist.get("media",0)}, baja={dist.get("baja",0)}')
        print()
        print('Modelos evaluados (CV 5-fold + embeddings semanticos):')
        for nombre, m in metricas.items():
            print(f'  {nombre}: accuracy = {m["accuracy_media"]} +/- {m["std"]}')
        print()
        print(f'Modelo seleccionado: {mejor}')
        print(f'Accuracy final: {acc:.4f} ({acc*100:.1f}%)')
        print()
        print('Tecnicas: SentenceTransformer embeddings, sin RAG.')
        print()
        print('[Texto generado por transformer]:')
        print(f'  {salida}')
        print('=' * 55)

agente3 = AgenteComunicador()
agente3.generar_reporte(resultado, df_raw)


AGENTE 3 - Comunicador
  Cargando modelo de generacion de texto...


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-proce

  [Generando reporte...]

REPORTE FINAL - Agente 3 Comunicador
Dataset analizado: 500 registros, 5 variables.
Valores faltantes tratados: 75 celdas.
Distribucion: alta=263, media=153, baja=59

Modelos evaluados (CV 5-fold + embeddings semanticos):
  Random Forest: accuracy = 1.0 +/- 0.0
  Gradient Boosting: accuracy = 1.0 +/- 0.0
  Logistic Regression: accuracy = 0.99 +/- 0.0089

Modelo seleccionado: Random Forest
Accuracy final: 1.0000 (100.0%)

Tecnicas: SentenceTransformer embeddings, sin RAG.

[Texto generado por transformer]:
  ML report: dataset with 500 employees. Best model: Random Forest, accuracy: 1.0000. Summary: factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors 